# Continued retries and PTRM on PPBench

Three experiments on the validation and golden splits of PPBench, with three seeds:

- up to 16 retries that continue the failed trajectory, without perturbation (TRM under the harness), with `y`
  reset or resampled, with `z` reset, resampled or perturbed by PTRM's noise, or both, and a restart without
  perturbation;
- PTRM's noise on `z` at five scales, in the continued retry that perturbs nothing else (validation split);
- PTRM rerun on our checkpoint: 100 rollouts of D = 16 and D = 48 steps at σ = 1.5, read with PTRM's own protocol and
  under the harness, the deterministic unroll of D = 48 steps, and the choice of σ on the validation split.

Set `CKPT_TRM`, `DATA_PATH` and `CACHE_DIR` below. Passes and rollouts are saved in `CACHE_DIR` and reloaded when the
notebook runs again. `PPBench_GAR` and `PPBench_z_instead_of_y` read the PTRM rollouts from there.

In [ ]:
!pip -q install einops pydantic argdantic
!test -d TRM || git clone -q https://github.com/SamsungSAILMontreal/TinyRecursiveModels TRM
!git -C TRM checkout -q c011037

In [ ]:
import os, sys, ast, json, dataclasses

import numpy as np
import pandas as pd
import torch
import yaml

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

CKPT_TRM = "/content/drive/MyDrive/GAR/ppbench/checkpoints/step_254961"  # with all_config.yaml next to it
DATA_PATH = "/content/drive/MyDrive/GAR/ppbench/data/ppbench-6type"  # built by PPBench_data_and_training
CACHE_DIR = "/content/drive/MyDrive/GAR/ppbench/cache"
GLOBAL_BATCH_SIZE = 768
DEVICE = "cuda"
REPORT_TYPES = ["sudoku", "lightup", "nurikabe", "heyawake", "tapa"]  # shakashaka is left out of the aggregates

sys.path.insert(0, os.path.abspath("TRM"))
from models.losses import ACTLossHead, IGNORE_LABEL_ID
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1
from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig

os.makedirs(CACHE_DIR, exist_ok=True)
pd.options.display.float_format = "{:.2f}".format

## Checkpoint, test sets and fillable cells

The test split has one set per puzzle type and split, `val_<type>` and `gold_<type>`, each loaded as one batch. The
model is compiled, as for the paper's numbers. PPBench has no blank token: the fillable cells of a puzzle are the
cells whose token does not belong to a family of clues fixed by the input (`STATIC_FAMILIES`). The check below
confirms, on the labels, that this mask contains every cell whose solution differs from the input and no unlabelled
cell.

In [ ]:
test_set = PuzzleDataset(PuzzleDatasetConfig(seed=0, dataset_paths=[DATA_PATH], global_batch_size=GLOBAL_BATCH_SIZE,
                                             test_set_mode=True, epochs_per_iter=1, rank=0, num_replicas=1),
                         split="test")
BATCHES = [(name, {k: v.to(DEVICE) for k, v in batch.items()}) for name, batch, _ in test_set]

with open(os.path.join(os.path.dirname(CKPT_TRM), "all_config.yaml")) as f:
    arch = yaml.safe_load(f)["arch"]
loss_type = arch.pop("loss")["loss_type"]
del arch["name"]
state = {k.replace("_orig_mod.", "", 1): v for k, v in torch.load(CKPT_TRM, map_location=DEVICE).items()}
config = dict(arch, batch_size=GLOBAL_BATCH_SIZE, vocab_size=test_set.metadata.vocab_size,
              seq_len=test_set.metadata.seq_len,
              num_puzzle_identifiers=state["model.inner.puzzle_emb.weights"].shape[0])
with torch.device(DEVICE):
    model = ACTLossHead(TinyRecursiveReasoningModel_ACTV1(config), loss_type=loss_type)
model.load_state_dict(state, strict=False, assign=True)
model = torch.compile(model.eval())  # halting is off: every block runs exactly T supervision steps
inner = model.model.inner
T = model.model.config.halt_max_steps  # supervision steps per block (16)
P = inner.puzzle_emb_len  # prefix positions before the 100 cells (16)

STATIC_FAMILIES = {"sudoku": {"g"}, "lightup": {"wall"}, "shakashaka": {"wall"}, "nurikabe": {"n"}, "tapa": {"c"},
                   "heyawake": set()}
with open(os.path.join(DATA_PATH, "vocab.json")) as f:
    VOCAB = {ast.literal_eval(k): v for k, v in json.load(f).items()}  # (type, clue, mark[, walls]) -> token
FILL_IDS = torch.tensor(sorted(v for (pid, clue, *_), v in VOCAB.items()
                               if (clue[0] if isinstance(clue, tuple) and clue else str(clue))
                               not in STATIC_FAMILIES.get(pid, set())), device=DEVICE, dtype=torch.int32)


def fillable(batch):
    """The fillable cells, read from the input tokens only."""
    return torch.isin(batch["inputs"], FILL_IDS.to(batch["inputs"].dtype))


for _, batch in BATCHES:
    labelled = batch["labels"] != IGNORE_LABEL_ID
    assert not (fillable(batch) & ~labelled).any()
    assert not ((batch["labels"] != batch["inputs"]) & labelled & ~fillable(batch)).any()


def split_sets(split):
    """The sets of a split that enter its aggregate, in the order used to concatenate puzzles."""
    return sorted(name for name, _ in BATCHES if name.startswith(split + "_") and name.split("_", 1)[1] in REPORT_TYPES)

## The harness

The standard pass runs one block of `T` supervision steps. The gate re-opens a puzzle whose halting logit `q` is
negative, and a retry runs one more block on it. The search stops at the first block with `q >= 0`, or after
`ROUNDS` retries, and returns the grid of highest `q` among the blocks run (the fallback).

A continued retry starts from the state of the failed block. At the block boundary, it can resample the fillable
cells of `y` (`z_H` in TRM's code) or reset them to `y_init`, and resample or reset to `z_init` the fillable cells,
all cells or all positions of `z` (`z_L`); it can also add PTRM's noise to `z` before every step of the block.
Resampled cells receive isotropic directions at the norm of `y_init` or `z_init`. `cold` restarts from the initial
states without perturbation, which replays the standard pass.

In [ ]:
Q_THR = 0.0  # the gate re-opens a puzzle when q < Q_THR, and a block with q >= Q_THR stops the search
SEEDS = [0, 1, 2]


def clone(x):
    """Deep copy of a TRM carry."""
    if torch.is_tensor(x):
        return x.clone()
    if dataclasses.is_dataclass(x):
        return type(x)(**{f.name: clone(getattr(x, f.name)) for f in dataclasses.fields(x)})
    if isinstance(x, dict):
        return {k: clone(v) for k, v in x.items()}
    return x


def correct(pred, batch):
    """Exact match on the labelled cells, as counted by TRM's evaluator, and the rows that have labels."""
    mask = batch["labels"] != IGNORE_LABEL_ID
    return (mask & (pred == batch["labels"])).sum(-1) == mask.sum(-1), mask.any(-1)


@torch.no_grad()
def run_block(batch, carry=None):
    """One block of T supervision steps; carry=None starts from the initial states (the standard pass)."""
    if carry is None:
        with torch.device(DEVICE):
            carry = model.initial_carry(batch)
    for _ in range(T):
        carry, _, _, out, _ = model(carry=carry, batch=batch, return_keys={"preds", "q_halt_logits"})
    return carry, out["preds"], out["q_halt_logits"].float()



ROUNDS = 16
SIGMA = 1.5  # PTRM's noise scale on PPBench, chosen with the sweep at the end of this notebook

# What a continued retry does to y and to z at the block boundary, and the noise it adds to z before every step.
ARMS = {"none": ("keep", "keep", 0.0),
        "randvec": ("resample", "keep", 0.0),
        "zvec": ("keep", "keep", SIGMA),
        "randvec_zvec": ("resample", "keep", SIGMA),
        "retract": ("reset", "keep", 0.0),
        "randz": ("keep", "resample", 0.0),
        "randz_all": ("keep", "resample all cells", 0.0),
        "randvec_randz": ("resample", "resample", 0.0),
        "zinit_cells": ("keep", "reset", 0.0),
        "zinit_all": ("keep", "reset all cells", 0.0),
        "zinit_full": ("keep", "reset all positions", 0.0),
        "randvec_zinit_cells": ("resample", "reset", 0.0),
        "randvec_zinit_all": ("resample", "reset all cells", 0.0),
        "randvec_zinit_full": ("resample", "reset all positions", 0.0),
        "cold": ("restart", "restart", 0.0)}


def directions(shape, gen, init):
    """Isotropic random directions, one per position, at the norm of `init`."""
    e = torch.randn(shape, generator=gen, device=DEVICE, dtype=torch.float32)
    return e / e.norm(dim=-1, keepdim=True).clamp_min(1e-9) * init.float().norm()


@torch.no_grad()
def retry(carry, fill, active, y_op, z_op, gen_y, gen_z):
    """Prepares a continued retry of the rows `active` at the block boundary."""
    zH, zL = carry.inner_carry.z_H, carry.inner_carry.z_L
    cells = (fill & active.view(-1, 1)).unsqueeze(-1)
    rows = active.view(-1, 1, 1)
    if y_op == "reset":
        zH[:, P:] = torch.where(cells, inner.H_init.to(zH.dtype), zH[:, P:])
    elif y_op == "resample":
        zH[:, P:] = torch.where(cells, directions(zH[:, P:].shape, gen_y, inner.H_init).to(zH.dtype), zH[:, P:])
    if z_op == "reset all positions":
        zL.copy_(torch.where(rows, inner.L_init.to(zL.dtype), zL))
    elif z_op.startswith("reset"):
        zL[:, P:] = torch.where(rows if z_op == "reset all cells" else cells, inner.L_init.to(zL.dtype), zL[:, P:])
    elif z_op.startswith("resample"):
        zL[:, P:] = torch.where(rows if z_op == "resample all cells" else cells,
                                directions(zL[:, P:].shape, gen_z, inner.L_init).to(zL.dtype), zL[:, P:])
    carry.halted.fill_(False)  # so that the next step continues from this state instead of resetting it
    carry.steps.zero_()


@torch.no_grad()
def noisy_block(batch, carry, sigma, active, gen):
    """One block of T steps, with noise N(0, sigma^2 I) added to z on the rows `active` before every step."""
    for _ in range(T):
        if sigma > 0:
            zL = carry.inner_carry.z_L
            e = torch.randn(zL.shape, generator=gen, device=zL.device, dtype=torch.float32)
            zL.copy_(torch.where(active.view(-1, 1, 1), (zL.float() + sigma * e).to(zL.dtype), zL))
        carry, _, _, out, _ = model(carry=carry, batch=batch, return_keys={"preds", "q_halt_logits"})
    return carry, out["preds"], out["q_halt_logits"].float()


@torch.no_grad()
def run_pass(arms, seed, sets=None):
    """The harness with each arm of `arms` (name -> (y_op, z_op, sigma)) on the test sets `sets` (all by default). Per
    set and per puzzle: the standard pass (ok0, q0, pred0) and, per arm, ok[:, r] (the returned grid is correct after r
    retries) and act[:, r] (the puzzle is open before retry r + 1)."""
    res = {}
    for b, (name, batch) in enumerate([(n, x) for n, x in BATCHES if sets is None or n in sets]):
        fill = fillable(batch)
        carry0, pred0, q0 = run_block(batch)
        ok0, valid = correct(pred0, batch)
        ok0 &= valid
        rows = valid.nonzero(as_tuple=True)[0]  # drops the padding rows of the batch
        res[name] = {"ok0": ok0[rows].cpu().numpy(), "q0": q0[rows].cpu().numpy(), "pred0": pred0[rows].cpu().numpy()}
        for arm, (y_op, z_op, sigma) in arms.items():
            gen_y = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b)
            gen_noise = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b + 7919)
            gen_z = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b + 15485863)
            carry = clone(carry0)
            best_p, best_q = pred0.clone(), q0.clone()
            active = valid & (q0 < Q_THR)
            ok, act = [ok0], []
            for r in range(ROUNDS):
                if arm == "cold" and r > 0:  # a restart without perturbation is run once
                    active = torch.zeros_like(active)
                act.append(active.clone())
                if active.any():
                    if arm == "cold":
                        carry, p, q = run_block(batch)
                    else:
                        retry(carry, fill, active, y_op, z_op, gen_y, gen_z)
                        carry, p, q = noisy_block(batch, carry, sigma, active, gen_noise)
                    better = active & (q > best_q)
                    best_p = torch.where(better.view(-1, 1), p, best_p)
                    best_q = torch.where(better, q, best_q)
                    active &= q < Q_THR
                ok.append(correct(best_p, batch)[0] & valid)
            res[name][arm + "__ok"] = torch.stack(ok, 1)[rows].cpu().numpy()
            res[name][arm + "__act"] = torch.stack(act, 1)[rows].cpu().numpy()
    return res

## The cache

In [ ]:
# The standard pass on every test set: every cached file must reproduce its halting logits and grids bit for bit.
REF = {}
for name, batch in BATCHES:
    _, pred0, q0 = run_block(batch)
    valid = correct(pred0, batch)[1]
    REF[name] = q0[valid].cpu().numpy(), pred0[valid].cpu().numpy()


def cached_pass(path, arms, seed, sets=None):
    """The pass of `arms` for `seed`, loaded from the cache or run and saved: per set, the standard pass and, per arm,
    the arrays ok and act."""
    sets = [name for name, _ in BATCHES if sets is None or name in sets]
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        arrays = {}
        for name, r in run_pass(arms, seed, sets).items():
            arrays.update({name + "__ok0": r.pop("ok0").astype(np.uint8), name + "__q0": r.pop("q0"),
                           name + "__pred0": r.pop("pred0").astype(np.int16)})
            arrays.update({"%s__%s" % (name, k): v.astype(np.uint8) for k, v in r.items()})
        np.savez_compressed(path, **arrays)
    f = np.load(path)
    same = all(np.array_equal(f[name + "__q0"], REF[name][0]) and np.array_equal(f[name + "__pred0"], REF[name][1])
               for name in sets)
    assert same, "%s comes from another numerical setup (GPU, PyTorch): delete it to recompute it" % path
    return {name: dict(ok0=f[name + "__ok0"].astype(bool), q0=f[name + "__q0"],
                       **{arm: dict(ok=f["%s__%s__ok" % (name, arm)].astype(bool),
                                    act=f["%s__%s__act" % (name, arm)].astype(bool)) for arm in arms})
            for name in sets}

## Continued retries

The arms are run in three passes, cached separately.

In [ ]:
PASS_ARMS = {"S%03d" % round(100 * SIGMA): ["none", "randvec", "zvec", "randvec_zvec", "cold"],
             "A": ["retract", "randz", "randz_all", "randvec_randz"],
             "B": ["zinit_cells", "zinit_all", "zinit_full", "randvec_zinit_cells", "randvec_zinit_all",
                   "randvec_zinit_full"]}
PASSES = []
for s in SEEDS:
    merged = {name: {} for name, _ in BATCHES}
    for tag, arms in PASS_ARMS.items():
        path = os.path.join(CACHE_DIR, "TRM_pass%s_seed%d.npz" % (tag, s))
        for name, r in cached_pass(path, {arm: ARMS[arm] for arm in arms}, s).items():
            merged[name].update(r)
    PASSES.append(merged)


def reference(sets):
    """Correctness and halting logit of the standard pass on the puzzles of `sets`."""
    return (np.concatenate([PASSES[0][name]["ok0"] for name in sets]),
            np.concatenate([PASSES[0][name]["q0"] for name in sets]))


def summary(passes, sets, arm):
    """After ROUNDS retries of `arm` on the puzzles of `sets`: failures recovered, accuracy, cost, successes lost."""
    ok0 = reference(sets)[0]
    ok = [np.concatenate([p[name][arm]["ok"][:, ROUNDS] for name in sets]) for p in passes]
    acc = [100 * v.mean() for v in ok]
    rec = [(~ok0 & v).sum() for v in ok]
    return {"recovered": np.mean(rec), "sd": np.std(rec, ddof=1), "per seed": tuple(int(x) for x in rec),
            "recovered (%)": 100 * np.mean(rec) / (~ok0).sum(), "sd (%)": 100 * np.std(rec, ddof=1) / (~ok0).sum(),
            "accuracy": np.mean(acc), "accuracy sd": np.std(acc, ddof=1),
            "cost": np.mean([1.0 + sum(p[name][arm]["act"][:, :ROUNDS].sum() for name in sets) / len(ok0)
                             for p in passes]),
            "lost": np.mean([(ok0 & ~v).sum() for v in ok])}


def gate_row(sets):
    """The standard pass and the gate on the puzzles of `sets`."""
    ok0, q0 = reference(sets)
    reopened = q0 < Q_THR
    return {"puzzles": len(ok0), "standard pass": 100 * ok0.mean(), "failures": (~ok0).sum(),
            "recall": 100 * (reopened & ~ok0).sum() / (~ok0).sum() if (~ok0).any() else np.nan,
            "false-positive rate": 100 * (reopened & ok0).sum() / ok0.sum() if ok0.any() else np.nan}

For each split: the standard pass and the gate per type and over the five types of the aggregate (recall: fraction
(%) of the failures that the gate re-opens; false-positive rate: fraction (%) of the successes that it re-opens).
Then, on the aggregate and after `ROUNDS` retries of each arm, the standard-pass failures recovered (mean, standard
deviation and value per seed, in counts and in % of the failures), the accuracy (%), the cost (blocks per puzzle)
and the standard-pass successes lost.

In [ ]:
GATE, LATTICE = {}, {}
for split in ("val", "gold"):
    sets = split_sets(split)
    GATE[split] = pd.DataFrame.from_dict({**{t: gate_row([split + "_" + t]) for t in REPORT_TYPES},
                                          "aggregate": gate_row(sets)}, orient="index")
    LATTICE[split] = pd.DataFrame.from_dict({arm: summary(PASSES, sets, arm) for arm in ARMS}, orient="index")
carry = run_block(BATCHES[0][1])[0]
print("Norm of a position after the standard pass: %.3f in y, %.3f in z (sqrt(d) = %.3f). Norm of y_init: %.3f, of "
      "z_init: %.3f." % (carry.inner_carry.z_H[:, P:].float().norm(dim=-1).mean(),
                         carry.inner_carry.z_L[:, P:].float().norm(dim=-1).mean(), inner.H_init.numel() ** 0.5,
                         inner.H_init.float().norm(), inner.L_init.float().norm()))
display(GATE["val"], LATTICE["val"], GATE["gold"], LATTICE["gold"])

## PTRM's noise in continued retries

The continued retry that perturbs nothing else, with PTRM's noise added to `z` before every step, at five scales.
Same columns as the table of arms, on the validation split after `ROUNDS` retries.

In [ ]:
rows = {}
for sigma in (0.2, 0.5, 1.0, 1.5, 2.0):
    arm = "dose_s%.2f" % sigma
    passes = [cached_pass(os.path.join(CACHE_DIR, "dose", "TRM_%s_seed%d.npz" % (arm, s)),
                          {arm: ("keep", "keep", sigma)}, s, split_sets("val")) for s in SEEDS]
    rows[sigma] = summary(passes, split_sets("val"), arm)
DOSE = pd.DataFrame.from_dict(rows, orient="index").rename_axis("sigma")
DOSE

## PTRM

A rollout starts from the initial states and runs `D` supervision steps, each preceded by noise N(0, σ² I) on every
position of `z`. On each test set, the rollouts run in batches of `m` copies of the set. Three checks come first, on
the first test set: without noise, a rollout of `T` steps reproduces the standard pass, alone or in copies, and a
rollout of `3T` steps reproduces the standard pass followed by two continued blocks.

In [ ]:
PTRM_K = 100
PTRM_D = [16, 48]
PAD = {"inputs": test_set.metadata.pad_id, "labels": IGNORE_LABEL_ID,
       "puzzle_identifiers": test_set.metadata.blank_identifier_id}


@torch.no_grad()
def rollout(batch, D, sigma, gen):
    """One PTRM rollout per row of `batch`: D supervision steps from the initial states, with noise on z before each
    step. Returns the grids and halting logits after the last step."""
    with torch.device(DEVICE):
        carry = model.initial_carry(batch)
    B = batch["inputs"].shape[0]
    # The initial states are written here rather than by the first step, which would overwrite the first noise.
    carry.inner_carry = inner.reset_carry(torch.ones(B, dtype=torch.bool, device=DEVICE), carry.inner_carry)
    carry.halted = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    carry.steps = torch.zeros(B, dtype=torch.int32, device=DEVICE)
    carry.current_data = {k: v.clone() for k, v in batch.items()}
    for _ in range(D):
        if sigma > 0:
            zL = carry.inner_carry.z_L
            e = torch.randn(zL.shape, generator=gen, device=zL.device, dtype=torch.float32)
            zL.copy_((zL.float() + sigma * e).to(zL.dtype))
        carry, _, _, out, _ = model(carry=carry, batch=batch, return_keys={"preds", "q_halt_logits"})
        carry.halted.fill_(False)
        carry.steps.zero_()
    return out["preds"], out["q_halt_logits"].float()


def pack(batch, rows, m):
    """m copies of the rows `rows` of `batch`, padded to the batch size as the test loader pads a batch."""
    out = {}
    for k, v in batch.items():
        copies = v[rows].repeat((m,) + (1,) * (v.ndim - 1))
        pad = torch.full((GLOBAL_BATCH_SIZE - len(copies),) + tuple(v.shape[1:]), PAD[k], dtype=v.dtype,
                         device=v.device)
        out[k] = torch.cat([copies, pad])
    return out


batch = BATCHES[0][1]
carry, pred0, q0 = run_block(batch)
rows = (batch["labels"] != IGNORE_LABEL_ID).any(-1).nonzero(as_tuple=True)[0]  # the puzzles, without padding
n, m, gen = len(rows), GLOBAL_BATCH_SIZE // len(rows), torch.Generator(device=DEVICE)
pred, q = rollout(pack(batch, rows, 1), T, 0.0, gen)
assert (pred[:n] == pred0[rows]).all() and (q[:n] == q0[rows]).all()
pred, q = rollout(pack(batch, rows, m), T, 0.0, gen)
assert all((pred[j * n:(j + 1) * n] == pred0[rows]).all() and (q[j * n:(j + 1) * n] == q0[rows]).all()
           for j in range(m))
for _ in range(2):  # two continued blocks after the standard pass
    carry.halted.fill_(False)
    carry.steps.zero_()
    carry, pred0, q0 = run_block(batch, carry)
pred, q = rollout(pack(batch, rows, 1), 3 * T, 0.0, gen)
assert (pred[:n] == pred0[rows]).all() and (q[:n] == q0[rows]).all()


@torch.no_grad()
def ptrm(K, D, sigma, seed, sets=None, tag=0, config=None):
    """K rollouts of D steps per puzzle of the test sets `sets` (all by default), loaded from the cache or run and
    saved: per set, the correctness and halting logit of each rollout, arrays [K, n]."""
    sets = [name for name, _ in BATCHES if sets is None or name in sets]
    path = os.path.join(CACHE_DIR, "ptrm", "TRM__%s__seed%d.npz" % (config or "K%d_D%d_s%.2f" % (K, D, sigma), seed))
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        arrays = {}
        for b, name in enumerate(sets):
            batch = dict(BATCHES)[name]
            rows = (batch["labels"] != IGNORE_LABEL_ID).any(-1).nonzero(as_tuple=True)[0]
            n, m = len(rows), GLOBAL_BATCH_SIZE // len(rows)
            gen = torch.Generator(device=DEVICE).manual_seed(seed * 1000003 + (100 * tag + b) * 7919 + D * 31
                                                             + round(sigma * 1000))
            packed = pack(batch, rows, m)
            labels = batch["labels"][rows]
            mask = labels != IGNORE_LABEL_ID
            ok, qs = np.zeros((K, n), bool), np.zeros((K, n), np.float32)
            for c in range(-(-K // m)):
                pred, q = rollout(packed, D, sigma, gen)
                for j in range(min(m, K - c * m)):
                    copy = slice(j * n, (j + 1) * n)
                    ok[c * m + j] = ((mask & (pred[copy] == labels)).sum(-1) == mask.sum(-1)).cpu().numpy()
                    qs[c * m + j] = q[copy].cpu().numpy()
            arrays.update({name + "__ok": ok, name + "__q": qs, name + "__ref_q0": REF[name][0]})
        np.savez_compressed(path, **arrays)
    f = np.load(path)
    assert all(np.array_equal(f[name + "__ref_q0"], REF[name][0]) for name in sets), \
        "%s comes from another numerical setup (GPU, PyTorch): delete it to recompute it" % path
    return {name: (f[name + "__ok"].astype(bool), f[name + "__q"]) for name in sets}


ROLLOUTS = {D: [ptrm(PTRM_K, D, SIGMA, s) for s in SEEDS] for D in PTRM_D}
UNROLL = ptrm(1, 48, 0.0, 0)  # TRM without noise for 48 steps

### With PTRM's own protocol and under the harness

The first `K'` rollouts of each configuration, with PTRM's selector (the rollout of highest `q`, *accuracy*) and the
oracle that accepts any correct rollout (*pass@K'*), then with the fallback, which also counts the standard pass as
a candidate and keeps it on ties. Under the harness, the gate re-opens the puzzles with `q < 0`; without the stopping
rule, the `K'` rollouts are all run on every re-opened puzzle, and with it they are read in order until one has
`q >= 0`. Cost in blocks per puzzle (one block per `T` steps), means and standard deviations over seeds; *lost*
counts the standard-pass successes that a configuration fails, and *recovered* the failures it solves.

In [ ]:
K_LIST = [k for k in (1, 5, 10, 25, 100) if k <= PTRM_K]


def read(ok, q, ok0, q0, k):
    """Correctness per puzzle of the first k rollouts: PTRM's selector, pass@k, and the selector with the fallback."""
    ok, q = ok[:k], q[:k]
    cols = np.arange(ok.shape[1])
    fallback = np.concatenate([ok0[None, :], ok])[np.concatenate([q0[None, :], q]).argmax(0), cols]
    return ok[q.argmax(0), cols], ok.any(0), fallback


def harness(ok, q, ok0, q0, k, D):
    """PTRM under the gate and the fallback, without and with the stopping rule: correctness per puzzle and cost."""
    ok, q = ok[:k], q[:k]
    n = ok.shape[1]
    cols, gate = np.arange(n), q0 < Q_THR
    cand_ok, cand_q = np.concatenate([ok0[None, :], ok]), np.concatenate([q0[None, :], q])
    gated = np.where(gate, cand_ok[cand_q.argmax(0), cols], ok0)
    hit = q >= Q_THR
    last = np.where(hit.any(0), hit.argmax(0), k - 1)
    read = np.concatenate([np.ones((1, n), bool), np.arange(k)[:, None] <= last[None, :]])
    stopped = np.where(gate, cand_ok[np.where(read, cand_q, -np.inf).argmax(0), cols], ok0)
    return gated, 1.0 + gate.mean() * (k * D / T), stopped, 1.0 + float(((last + 1) * gate).sum()) * D / T / n


def accuracy(ok, ok0):
    """Mean and standard deviation over seeds of the accuracy (%), and means of the successes lost and failures
    recovered."""
    acc = [100 * v.mean() for v in ok]
    return (np.mean(acc), np.std(acc, ddof=1), np.mean([(ok0 & ~v).sum() for v in ok]),
            np.mean([(~ok0 & v).sum() for v in ok]))


PTRM_OWN, PTRM_HARNESS = {}, {}
for split in ("val", "gold"):
    sets = split_sets(split)
    ok0, q0 = reference(sets)
    own, harn = {}, {}
    for D in PTRM_D:
        rollouts = [[np.concatenate([r[name][i] for name in sets], 1) for i in (0, 1)] for r in ROLLOUTS[D]]
        for k in K_LIST:
            sel, pas, fb = zip(*(read(ok, q, ok0, q0, k) for ok, q in rollouts))
            (a, sd, lost, rec), (pa, psd, _, _), (fa, fsd, flost, _) = (accuracy(sel, ok0), accuracy(pas, ok0),
                                                                         accuracy(fb, ok0))
            own[(k, D)] = {"cost": k * D / T, "accuracy": a, "sd": sd, "pass@K'": pa, "pass sd": psd, "lost": lost,
                           "recovered": rec, "fallback accuracy": fa, "fallback sd": fsd, "fallback lost": flost}
            gated, gated_cost, stopped, stopped_cost = zip(*(harness(ok, q, ok0, q0, k, D) for ok, q in rollouts))
            (a, sd, _, _), (sa, ssd, _, _) = accuracy(gated, ok0), accuracy(stopped, ok0)
            harn[(k, D)] = {"cost": gated_cost[0], "accuracy": a, "sd": sd, "stopping rule cost": np.mean(stopped_cost),
                            "stopping rule accuracy": sa, "stopping rule sd": ssd}
    unroll = np.concatenate([UNROLL[name][0][0] for name in sets])
    own[("unroll", 48)] = {"cost": 48 / T, "accuracy": 100 * unroll.mean(), "lost": (ok0 & ~unroll).sum(),
                           "recovered": (~ok0 & unroll).sum()}
    PTRM_OWN[split] = pd.DataFrame.from_dict(own, orient="index").rename_axis(["K'", "D"])
    PTRM_HARNESS[split] = pd.DataFrame.from_dict(harn, orient="index").rename_axis(["K'", "D"])
display(PTRM_OWN["val"], PTRM_HARNESS["val"], PTRM_OWN["gold"], PTRM_HARNESS["gold"])

### The noise scale

Pass@25 and best-Q@25 (%) on the validation split, with K = 25 rollouts of D = 16 steps per scale: means and
standard deviations over seeds.

In [ ]:
sets = split_sets("val")
rows = {}
for sigma in (0.0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5):
    best, passed = [], []
    for s in SEEDS:
        r = ptrm(25, 16, sigma, s, sets, tag=7, config="sweep_s%.2f" % sigma)
        ok, q = np.concatenate([r[name][0] for name in sets], 1), np.concatenate([r[name][1] for name in sets], 1)
        best.append(100 * ok[q.argmax(0), np.arange(ok.shape[1])].mean())
        passed.append(100 * ok.any(0).mean())
    rows[sigma] = {"pass@25": np.mean(passed), "pass sd": np.std(passed, ddof=1),
                   "best-Q@25": np.mean(best), "best-Q sd": np.std(best, ddof=1)}
SIGMA_SWEEP = pd.DataFrame.from_dict(rows, orient="index").rename_axis("sigma")
SIGMA_SWEEP